# AAPL Price-Only Regressor (30-Year History)
Trains an XGBoost regressor on lagged returns / closing prices to predict
next-day (T+1) % return. This model is the **primary/base** signal that will
later be combined with the news+sentiment impact model (multiplier/adjustment).


In [1]:
# CELL 1: IMPORTS AND SETUP
import pandas as pd
import numpy as np
import yfinance as yf
import joblib
import os
import math
import warnings
from datetime import datetime
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')

CHECKPOINT_DIR = "./checkpoints2/"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("✓ Imports successful")
print(f"✓ Checkpoint directory: {CHECKPOINT_DIR}")


✓ Imports successful
✓ Checkpoint directory: ./checkpoints2/


In [2]:
# CELL 2: DOWNLOAD 30-YEAR AAPL DATA
from datetime import datetime
import pandas as pd
import yfinance as yf

print("\n" + "="*80)
print("STEP 1: Download AAPL 30-year price history")
print("="*80)

# Calculate date range (30 years back from today)
end_date = datetime.now()
start_date = end_date.replace(year=end_date.year - 30)

print(f"Fetching AAPL from {start_date.date()} to {end_date.date()}...")

# Download data from yfinance
prices = yf.download("AAPL", start=start_date, end=end_date, progress=False, auto_adjust=False)

# Handle MultiIndex columns if present
if isinstance(prices.columns, pd.MultiIndex):
    prices.columns = prices.columns.get_level_values(0)

# Explicitly name the index 'Date' so it resets cleanly into a 'Date' column
prices.index.name = 'Date'
prices = prices.reset_index()

# Normalize column names robustly
rename_map = {}
for col in prices.columns:
    cl = col.lower()
    if 'date' in cl or cl == 'index':  # Catch 'index' as a fallback
        rename_map[col] = 'Date'
    elif cl == 'close' or cl.startswith('close_'):
        rename_map[col] = 'Close'
    elif cl == 'open' or cl.startswith('open_'):
        rename_map[col] = 'Open'
    elif cl == 'high' or cl.startswith('high_'):
        rename_map[col] = 'High'
    elif cl == 'low' or cl.startswith('low_'):
        rename_map[col] = 'Low'
    elif cl == 'volume' or cl.startswith('volume_'):
        rename_map[col] = 'Volume'

prices = prices.rename(columns=rename_map)

# Clean up datetime formats and strip timezones if they exist
prices['Date'] = pd.to_datetime(prices['Date'])
if prices['Date'].dt.tz is not None:
    prices['Date'] = prices['Date'].dt.tz_localize(None)

# Final sorting and cleanup
prices = prices.sort_values('Date').reset_index(drop=True)
prices = prices[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].dropna().reset_index(drop=True)

print(f"✓ Fetched {len(prices):,} trading days")
print(f"✓ Date range: {prices['Date'].min().date()} → {prices['Date'].max().date()}")
prices.head()


STEP 1: Download AAPL 30-year price history
Fetching AAPL from 1996-06-16 to 2026-06-16...
✓ Fetched 7,546 trading days
✓ Date range: 1996-06-17 → 2026-06-12


Price,Date,Open,High,Low,Close,Volume
0,1996-06-17,0.215402,0.215402,0.210938,0.210938,112929600
1,1996-06-18,0.210938,0.212054,0.202009,0.203125,223227200
2,1996-06-19,0.206473,0.208705,0.202009,0.206473,134467200
3,1996-06-20,0.208705,0.208705,0.200893,0.203125,147089600
4,1996-06-21,0.204241,0.204241,0.199777,0.202009,161851200


In [3]:
# CELL 3: FEATURE ENGINEERING (LAGGED RETURNS / CLOSES)
print("\n" + "="*80)
print("STEP 2: Build lagged-return features")
print("="*80)

price_df = prices.copy()

# Daily return (%)
price_df['ret_1d'] = price_df['Close'].pct_change(1) * 100

# Lagged returns as features (use past N days of returns to predict next-day move)
N_LAGS = 5
for lag in range(1, N_LAGS + 1):
    price_df[f'lag_ret_{lag}'] = price_df['ret_1d'].shift(lag)

# Lagged closes (normalized as pct change from N days ago) — extra signal
for lag in range(1, N_LAGS + 1):
    price_df[f'lag_close_pct_{lag}'] = price_df['Close'].pct_change(lag) * 100

# Target: next-day return (T+1)
price_df['target_ret_T1'] = price_df['ret_1d'].shift(-1)

PRICE_FEATURE_COLS = (
    [f'lag_ret_{i}' for i in range(1, N_LAGS + 1)] +
    [f'lag_close_pct_{i}' for i in range(1, N_LAGS + 1)]
)

price_model_df = price_df.dropna(subset=PRICE_FEATURE_COLS + ['target_ret_T1']).reset_index(drop=True)

print(f"✓ Feature columns: {PRICE_FEATURE_COLS}")
print(f"✓ Training rows after dropna: {len(price_model_df):,}")
price_model_df[['Date', 'Close', 'target_ret_T1'] + PRICE_FEATURE_COLS].head()



STEP 2: Build lagged-return features
✓ Feature columns: ['lag_ret_1', 'lag_ret_2', 'lag_ret_3', 'lag_ret_4', 'lag_ret_5', 'lag_close_pct_1', 'lag_close_pct_2', 'lag_close_pct_3', 'lag_close_pct_4', 'lag_close_pct_5']
✓ Training rows after dropna: 7,539


Price,Date,Close,target_ret_T1,lag_ret_1,lag_ret_2,lag_ret_3,lag_ret_4,lag_ret_5,lag_close_pct_1,lag_close_pct_2,lag_close_pct_3,lag_close_pct_4,lag_close_pct_5
0,1996-06-25,0.184152,-3.636677,-1.657356,-0.549412,-1.621516,1.648243,-3.703935,-7.303392,-8.839705,-9.340550,-10.810608,-9.340550
1,1996-06-26,0.177455,3.773923,-7.303392,-1.657356,-0.549412,-1.621516,1.648243,-3.636677,-10.674469,-12.154911,-12.637542,-14.054138
2,1996-06-27,0.184152,1.818059,-3.636677,-7.303392,-1.657356,-0.549412,-1.621516,3.773923,0.000000,-7.303392,-8.839705,-9.340550
3,1996-06-28,0.187500,2.380800,3.773923,-3.636677,-7.303392,-1.657356,-0.549412,1.818059,5.660594,1.818059,-5.618113,-7.182357
4,1996-07-01,0.191964,-2.325436,1.818059,3.773923,-3.636677,-7.303392,-1.657356,2.380800,4.242144,8.176162,4.242144,-3.371069


In [4]:
# CELL 4: TRAIN / TEST SPLIT (TIME-BASED)
print("\n" + "="*80)
print("STEP 3: Time-based train/test split")
print("="*80)

# Time-ordered split — no shuffling, avoids look-ahead leakage
split_idx = int(len(price_model_df) * 0.85)
train_price = price_model_df.iloc[:split_idx]
test_price  = price_model_df.iloc[split_idx:]

X_train = train_price[PRICE_FEATURE_COLS]
y_train = train_price['target_ret_T1']
X_test  = test_price[PRICE_FEATURE_COLS]
y_test  = test_price['target_ret_T1']

print(f"✓ Train rows: {len(train_price):,} ({train_price['Date'].min().date()} → {train_price['Date'].max().date()})")
print(f"✓ Test rows : {len(test_price):,} ({test_price['Date'].min().date()} → {test_price['Date'].max().date()})")



STEP 3: Time-based train/test split
✓ Train rows: 6,408 (1996-06-25 → 2021-12-07)
✓ Test rows : 1,131 (2021-12-08 → 2026-06-11)


In [5]:
# CELL 5: TRAIN XGBOOST REGRESSOR
print("\n" + "="*80)
print("STEP 4: Train XGBoost price-only regressor")
print("="*80)

price_model = XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

price_model.fit(X_train, y_train)

print("✓ Model trained")



STEP 4: Train XGBoost price-only regressor
✓ Model trained


In [6]:
# CELL 6: EVALUATE
print("\n" + "="*80)
print("STEP 5: Evaluate on test set")
print("="*80)

preds_test = price_model.predict(X_test)

test_mae  = mean_absolute_error(y_test, preds_test)
test_rmse = math.sqrt(mean_squared_error(y_test, preds_test))

# Baseline: naive "predict zero change" benchmark
baseline_mae = mean_absolute_error(y_test, np.zeros_like(y_test))

print(f"✓ Test MAE          : {test_mae:.4f}")
print(f"✓ Test RMSE         : {test_rmse:.4f}")
print(f"✓ Baseline MAE (0)  : {baseline_mae:.4f}")

# Bucket evaluation (HIGH / NEUTRAL / LOW) — same thresholds used in combination step
HIGH_THRESHOLD = 0.5
LOW_THRESHOLD  = -0.5

def bucket_from_return(pct_return):
    if pct_return > HIGH_THRESHOLD:
        return "HIGH"
    elif pct_return < LOW_THRESHOLD:
        return "LOW"
    else:
        return "NEUTRAL"

eval_df = pd.DataFrame({
    'date': test_price['Date'].values,
    'actual_return': y_test.values,
    'predicted_return': preds_test
})
eval_df['actual_bucket']    = eval_df['actual_return'].apply(bucket_from_return)
eval_df['predicted_bucket'] = eval_df['predicted_return'].apply(bucket_from_return)

bucket_accuracy = (eval_df['actual_bucket'] == eval_df['predicted_bucket']).mean()
print(f"✓ Bucket accuracy   : {bucket_accuracy:.2%}")
print("\nPredicted bucket distribution:")
print(eval_df['predicted_bucket'].value_counts())
print("\nActual bucket distribution:")
print(eval_df['actual_bucket'].value_counts())

eval_df.head(10)



STEP 5: Evaluate on test set
✓ Test MAE          : 1.2659
✓ Test RMSE         : 1.7910
✓ Baseline MAE (0)  : 1.2520
✓ Bucket accuracy   : 31.30%

Predicted bucket distribution:
predicted_bucket
NEUTRAL    1070
HIGH         50
LOW          11
Name: count, dtype: int64

Actual bucket distribution:
actual_bucket
HIGH       417
LOW        367
NEUTRAL    347
Name: count, dtype: int64


,date,actual_return,predicted_return,actual_bucket,predicted_bucket
0,2021-12-08,-0.297010,0.384913,NEUTRAL,NEUTRAL
1,2021-12-09,2.801329,0.295152,HIGH,NEUTRAL
2,2021-12-10,-2.067424,0.270107,LOW,NEUTRAL
3,2021-12-13,-0.802324,0.291760,LOW,NEUTRAL
4,2021-12-14,2.850916,-0.114374,HIGH,NEUTRAL
5,2021-12-15,-3.926385,0.265226,LOW,NEUTRAL
6,2021-12-16,-0.650177,-0.082146,LOW,NEUTRAL
7,2021-12-17,-0.812200,0.323589,LOW,NEUTRAL
8,2021-12-20,1.908692,0.195941,HIGH,NEUTRAL
9,2021-12-21,1.531877,-0.011476,HIGH,NEUTRAL


In [7]:
# CELL 7: FEATURE IMPORTANCE
print("\n" + "="*80)
print("STEP 6: Feature importance")
print("="*80)

importance_df = pd.DataFrame({
    'feature': PRICE_FEATURE_COLS,
    'importance': price_model.feature_importances_
}).sort_values('importance', ascending=False)

print(importance_df.to_string(index=False))



STEP 6: Feature importance
        feature  importance
lag_close_pct_1    0.120338
      lag_ret_2    0.105506
      lag_ret_5    0.101325
lag_close_pct_2    0.100555
lag_close_pct_5    0.098774
      lag_ret_4    0.098114
lag_close_pct_4    0.097608
      lag_ret_3    0.095263
      lag_ret_1    0.092297
lag_close_pct_3    0.090219


In [8]:
# CELL 8: SAVE MODEL + ARTIFACTS
print("\n" + "="*80)
print("STEP 7: Save model and artifacts")
print("="*80)

price_model_path  = f"{CHECKPOINT_DIR}price_model.pkl"
feature_cols_path = f"{CHECKPOINT_DIR}price_feature_cols.pkl"
metrics_path      = f"{CHECKPOINT_DIR}price_model_metrics.pkl"

joblib.dump(price_model, price_model_path)
joblib.dump(PRICE_FEATURE_COLS, feature_cols_path)
joblib.dump({'test_mae': test_mae, 'test_rmse': test_rmse, 'bucket_accuracy': bucket_accuracy},
            metrics_path)

# Also save the full feature-engineered dataset for later reuse / combination step
price_model_df.to_csv(f"{CHECKPOINT_DIR}aapl_30yr_price_features.csv", index=False)

print(f"✓ Saved model to       {price_model_path}")
print(f"✓ Saved feature cols   {feature_cols_path}")
print(f"✓ Saved metrics        {metrics_path}")
print(f"✓ Saved feature dataset {CHECKPOINT_DIR}aapl_30yr_price_features.csv")
print("\n" + "="*80)
print("✓ DONE — price model ready for combination with news/sentiment model")
print("="*80)



STEP 7: Save model and artifacts
✓ Saved model to       ./checkpoints2/price_model.pkl
✓ Saved feature cols   ./checkpoints2/price_feature_cols.pkl
✓ Saved metrics        ./checkpoints2/price_model_metrics.pkl
✓ Saved feature dataset ./checkpoints2/aapl_30yr_price_features.csv

✓ DONE — price model ready for combination with news/sentiment model
